# Transductive fusion

This notebook presents the paper's three-panel comparison of MeSH-only,
topology-only, fusion, and two DDI-network baselines. The default cells validate
and display the supplied aggregate results; no private data or training is needed.

See the [setup instructions](../../README.md) for dependencies.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../data/README.md) and
[DataPreparation.ipynb](../../data/DataPreparation.ipynb), you can rerun the
experiment and regenerate the figure using the functions in
[`src/dtpkg/transductive/`](../../src/dtpkg/transductive/).

## Paths and run options

Replace each `...` in the next cell with your own quoted path, for example
`DATA_DIR = "/path/to/private/data"`. `PROJECT_ROOT` is the checkout folder;
`DATA_DIR` follows the [data guide](../../data/README.md). In the fusion notebooks,
also supply `GRAPH_PATH`, the complete path to your GraphML file.
Then adjust the derived input and output paths as needed and rerun the cells.
The path cell intentionally requires completion before execution; no config-file
or environment-variable changes are needed. The default run uses only the public results.


In [ ]:
from pathlib import Path
import sys

# Replace each ... with your own quoted path or Path(...).
PROJECT_ROOT = ...  # DTP-KG checkout folder
DATA_DIR = ...  # Local data folder; see data/README.md
GRAPH_PATH = ...  # Complete path to the canonical GraphML network

# Check and normalize the paths supplied above.
if any(path is ... for path in (PROJECT_ROOT, DATA_DIR, GRAPH_PATH)):
    raise ValueError("Replace ... in PROJECT_ROOT, DATA_DIR, GRAPH_PATH with your local paths.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DATA_DIR = Path(DATA_DIR).expanduser().resolve()
GRAPH_PATH = Path(GRAPH_PATH).expanduser().resolve()

# Public aggregate inputs and regenerated figures.
EXPERIMENT = PROJECT_ROOT / "experiments" / "04_transductive_fusion"
TABLES = EXPERIMENT / "tables"
FIGURES = EXPERIMENT / "figures"
OUTPUT_DIR = FIGURES / "reproduced"

# Inputs and outputs for optional private preparation/training.
PRIVATE_RUN = EXPERIMENT / "results" / "transductive_fusion"

REPLOT = False
RUN_TRAINING = False


In [2]:
from IPython.display import display, Image
from dtpkg.transductive.reporting import load_figure_tables, ORDER, SHORT, FIVE_MODELS
from dtpkg.transductive.plot_three_panel import render


## Inputs and comparison

The MeSH representation is the Intermediate scope selected in experiment 03,
with 128 TF-IDF/SVD values per drug. Topology uses 12 features from radius-two
neighbourhoods of the canonical DDI/DPI/PPI network, with spectral Katz scaling,
no closeness cutoff, and no betweenness feature.

| Method | Input |
| --- | --- |
| MeSH-only | Symmetric pair features from MeSH embeddings |
| Topology-only | Symmetric pair features from standardized topology |
| Fusion | Per-drug MeSH and topology encoders combined by a learned vector gate, then symmetric pair features |
| DDI common neighbours | Number of common neighbours in the masked drug-only graph |
| DDI degree product | Product of the two degrees in the masked drug-only graph |

This is **pair holdout**: drugs may occur in both training and test pairs. The
saved cohort contains 6,500 pairs, balanced between positives and curated
negatives. Validation and test positive DDI edges are removed before each fold's
topology extraction and graph scoring. Other known DDIs remain available.
Fixed MeSH embeddings are prepared globally.

## Training and evaluation protocol

Five repeated five-fold partitions give **25 split bundles**. Each bundle holds
three neural models, giving 75 neural fits, alongside the two graph heuristics.
All methods share evaluation pairs and partitions. Within each outer development
partition, 15% is reserved for validation. Training positives are resampled each
epoch from the eligible pool, excluding held-out pairs, with a 1:1 ratio to
training negatives.

Neural models use Adam (learning rate 0.001, weight decay 0.00001), at most 100
epochs, and patience 10. Checkpoints minimize **validation binary cross-entropy**.
MeSH-only uses batches of 128; fusion and topology-only update once per pair.
The MeSH baseline's scaler is fitted on epoch-zero training pair features;
topology-only uses mean/std over unique drugs in epoch-zero training pairs.
Fusion passes raw MeSH/topology inputs to learned encoders. These settings match
the saved experiment and differ from the MeSH-scope experiment's scaling policy.

Neural classification uses threshold 0.5. Graph-score thresholds maximize
inner-validation F1; tied maxima choose the highest threshold. All outer-test
scores use the validation-selected models or thresholds.

## Validate the public aggregate results

The public tables retain 750 category/model/fold rows, 60 AUROC/F1 summaries,
and all 96 Fusion-versus-comparator contrasts. The loader recomputes the means,
pointwise corrected-CV intervals, raw p-values, and Holm correction to verify
the figure inputs. It does not pool predictions across fitted models.

Low, Mid, and Deep identify a drug's deepest assigned MeSH level (1–5, 6–7,
and 8–10), not the representation scope. LL/LM/LD/MM/MD/DD label unordered
pairs of these categories.

In [3]:
summary, comparisons = load_figure_tables(TABLES)
assert len(summary) == 60
assert len(comparisons) == 96
assert comparisons["family_size"].eq(96).all()

model_names = {
    "baseline": "MeSH-only", "topo_only": "Topology-only", "fusion": "Fusion",
    "common_neighbors": "DDI common neighbours", "degree_product": "DDI degree product",
}
for metric, label in (("auc", "AUROC"), ("f1", "F1")):
    values = summary[summary.metric == metric].pivot(index="category", columns="model", values="estimate")
    values = values.reindex(index=ORDER, columns=FIVE_MODELS).rename(index=SHORT, columns=model_names)
    values.index.name = label
    display(values.round(4))

model,MeSH-only,Topology-only,Fusion,DDI common neighbours,DDI degree product
AUROC,,,,,
LL,0.9664,0.9508,0.9887,0.9163,0.7801
LM,0.9580,0.9366,0.9885,0.8715,0.7516
LD,0.9715,0.9505,0.9919,0.8770,0.7252
MM,0.9664,0.9404,0.9881,0.9059,0.8030
MD,0.9650,0.9466,0.9855,0.9028,0.7934
DD,0.9643,0.9447,0.9849,0.9472,0.8474


model,MeSH-only,Topology-only,Fusion,DDI common neighbours,DDI degree product
F1,,,,,
LL,0.8686,0.8416,0.9532,0.7817,0.6583
LM,0.8657,0.8150,0.9403,0.7356,0.6221
LD,0.9096,0.8667,0.9486,0.7660,0.6842
MM,0.9037,0.8595,0.9506,0.8211,0.7431
MD,0.9253,0.8949,0.9585,0.8520,0.8082
DD,0.9441,0.9287,0.9719,0.9355,0.9092


## Paper figure

![Transductive fusion three-panel figure](figures/transductive_fusion_three_panel.png)

Panels (a) and (b) show five-method AUROC and F1; panel (c) shows paired
Fusion-minus-MeSH differences in AUROC, F1, precision, and recall.
The rows beneath the curves report Fusion-minus-comparator differences.
[Full caption](figures/transductive_fusion_three_panel_caption.txt).

The current exploratory Holm family includes **96 comparisons**: four Fusion
contrasts × six categories × four metrics. This reporting family was revised
after inspection of the results; it is distinct from the original planned
60-test analysis. Stars use Holm-adjusted p-values, while daggers mark
unadjusted p < 0.05 with adjusted p ≥ 0.05. Error bars are pointwise intervals,
not simultaneous intervals adjusted with Holm.

## Optional: redraw the figure

Set `REPLOT = True` above to write the PNG, caption, plotting tables, and
provenance to `OUTPUT_DIR` (by default, ignored `figures/reproduced/`). The supplied reference and aggregate
input tables are preserved.

In [4]:
if REPLOT:
    reproduced = render(TABLES, OUTPUT_DIR)
    display(Image(filename=str(reproduced)))

## Optional: train from private inputs

Use the shared tables from [data preparation](../../data/DataPreparation.ipynb),
the graph from [network construction](../02_network_construction/NetworkConstruction.ipynb),
and the Intermediate embedding from [MeSH scope](../03_mesh_scope/MeSHScope.ipynb).
Set `DATA_DIR`, `GRAPH_PATH`, and `PRIVATE_RUN` in the paths cell above.

Set `RUN_TRAINING = True` to run the full protocol. `PRIVATE_RUN` must be new or
empty; its checkpoint bundles, predictions, per-drug gates, and caches remain
private in the ignored `results/` directory. `topology_jobs` controls parallel
feature extraction. The function returns the completed run directory.

In [5]:
if RUN_TRAINING:
    from dtpkg.transductive.experiment import run_transductive

    run_transductive(PRIVATE_RUN, data_dir=DATA_DIR, graph_path=GRAPH_PATH,
                    topology_jobs=16)
    render(PRIVATE_RUN, OUTPUT_DIR / "new_run")

## Interpretation and reproduction limits

This experiment compares complete pipelines with different optimization and
scaling policies; it does not isolate the gating mechanism or the contribution
of DDI edges. Pair splits share drugs, and approximate corrected-CV inference
does not remove all network dependence. The balanced, curated-negative cohort
also differs from deployment prevalence. Non-significance does not establish
equivalence.

DrugBank-derived data and record-level outputs are not distributed. The supplied
figure and aggregate tables preserve the completed study run, with derivations
recorded in [snapshot provenance](tables/snapshot_provenance.json). Packaging
these files did not retrain the study.